In [ ]:
import numpy as np
import glob
from scipy.special import jn # Bessel Functions of the First Kind (Jn)
from scipy.special import yv # Bessel Function of the Second Kind (Yn)
from scipy.special import hankel1
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.size" : 15,
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "mathtext.fontset": "stix",
})


In [ ]:
c_s = 1.0 / np.sqrt(3.0)
f0 = 0.05
rhoAmp = 0.01
rhoRef = 1.0

# vis = 0.0               # tau=0.5
# vis = 0.033333333       # tau=0.6
vis = 0.166666667       # tau=1.0


Lx = 201
Ly = 201

step = 200

folderVec = [

    # tau=0.5
    # "/Users/maggie/repo/LBM-Program/amrlbm/bin/acoustic_validation/pointSourceTest_cml_w1_1_tau05/probes/probeX",
    # "/Users/maggie/repo/LBM-Program/amrlbm/bin/acoustic_validation/pointSourceTest_cml_w1_w0_tau05/probes/probeX",
    # "/Users/maggie/repo/LBM-Program/amrlbm/bin/acoustic_validation/pointSourceTest_srt_tau05/probes/probeX",
    # "/Users/maggie/repo/LBM-Program/amrlbm/bin/acoustic_validation/pointSourceTest_srt_smag_tau05/probes/probeX",

    # tau=0.6
    # "/Users/maggie/repo/LBM-Program/amrlbm/bin/acoustic_validation/pointSourceTest_cml_w1_1_tau06/probes/probeX",       #! exploded
    # "/Users/maggie/repo/LBM-Program/amrlbm/bin/acoustic_validation/pointSourceTest_cml_w1_w0_tau06/probes/probeX",
    # "/Users/maggie/repo/LBM-Program/amrlbm/bin/acoustic_validation/pointSourceTest_srt_tau06/probes/probeX",
    # "/Users/maggie/repo/LBM-Program/amrlbm/bin/acoustic_validation/pointSourceTest_srt_tau06_test/probes/probeX",

    # tau=1.0
    # "/Users/maggie/repo/LBM-Program/amrlbm/bin/acoustic_validation/pointSourceTest_cml_w1_1_tau1/probes/probeX",       #! exploded
    # "/Users/maggie/repo/LBM-Program/amrlbm/bin/acoustic_validation/pointSourceTest_cml_w1_w0_tau1/probes/probeX",       #! exploded
    # "/Users/maggie/repo/LBM-Program/amrlbm/bin/acoustic_validation/pointSourceTest_srt_tau1/probes/probeX",
    "/Users/maggie/repo/LBM-Program/amrlbm/bin/acoustic_validation/pointSourceTest_srt_tau1_test/probes/probeX",
]

labelVec = [
    # r'LBM Cumulant $\omega_1=1.0$',
    # r'LBM Cumulant $\omega_1=\omega_0$',
    'LBM BGK',
    # 'LBM BGK (3rd equilibrium, smag)',
    # 'LBM BGK (post-collision & pre-streaming)',
    # 'LBM BGK (post-streaming)',
]

colorVec = [
    # "#910000",
    # "#119100",
    "#001AFF",
    '#FF5733',
    "#B700FF",
]
lstyleVec = [
    # '--',
    '-.',
    ':',
    '--'
]

In [ ]:

visBulk = 2.0 / 3.0 * vis
tau = vis / c_s**2 + 0.5
tau_vi = 1.0 / c_s**2 * (4.0 / 3.0 * vis + visBulk)

omega0 = 2.0 * np.pi * f0
k0 = omega0 / c_s

omega = omega0 * (1.0 - 1.0 / 8.0 * (omega0 * tau_vi)**2)
alpha_t = omega0**2 * tau_vi * 0.5 #! book P500 (12.13b)
alpha_x = k0 * omega0 * tau_vi * 0.5
k = k0 / (1.0 + 3.0 / 8.0 * (omega0 * tau_vi)**2)
k_hat = k - 1j * alpha_x

print(f"c_s = {c_s:.6f}")
print(f"f0 = {f0:6f}")
print(f"vis_bulk = {visBulk:.6f}")
print(f"tau = {tau:.6f}")
print(f"tau_vi = {tau_vi:.6f}")
print(f"omega = {omega:.6f}")
print(f"alpha_x = {alpha_x:.6f}")
print(f"k = {k:.6f}")


In [ ]:
def read_col_probe(filePath, colIdx, skipHeader=2):
    with open(filePath, 'r') as f:
        vec = np.genfromtxt(f, skip_header=skipHeader, usecols=colIdx)
        vec = vec[~np.isnan(vec)]
    f.close()
    return vec
def get_probe_coord_value(filePath, coordIdx):
    with open(filePath, 'r') as f:
        header = f.readline()
        coordValue = float(header.split('(')[1].split(')')[0].split(',')[coordIdx])
    return coordValue

coordMap = { 'x': 0, 'y': 1, 'z': 2 }
coordIdx = coordMap['x']
colIdx = 2


coordVecVec = []
dataVecVec = []
for case in range(0, len(folderVec)):
    fileList = sorted(glob.glob(f"{folderVec[case]}/probe*.txt"))
    coordVec = []
    dataVec = []
    for filePath in fileList:
        coordValue = get_probe_coord_value(filePath, coordIdx)
        # coordVec.append(coordValue)
        coordVec.append(coordValue + 0.5)
        stepCol = read_col_probe(filePath, 0, 2)
        dataCol = read_col_probe(filePath, colIdx)
        targetStepIdx = int(np.abs(stepCol - (step + 1)).argmin())
        dataVec.append(dataCol[targetStepIdx] - rhoRef)
    coordVecVec.append(coordVec)
    dataVecVec.append(dataVec)


In [ ]:

x = np.linspace(0.0, 100.0, 300)
xSafe = np.where(x == 0, 1, x)
##!
j0 = jn(0, k_hat * xSafe)
y0 = yv(0, k_hat * xSafe)
h0 = j0 - 1j * y0

# amp = 0.135 * rhoAmp * np.exp(-1j * 2.0 * np.pi / 3.0) # tau=1.0 #! Viggen 2009
# amp = 0.15 * rhoAmp * np.exp(-1j * 2.0 * np.pi / 3.0)   # tau=0.6 #! Viggen 2009

xRef = 0.0001
rhoPrimeRef = rhoAmp

j0Ref = jn(0, k_hat * xRef)
y0Ref = yv(0, k_hat * xRef)
h0Ref = j0Ref - 1j * y0Ref
amp = rhoPrimeRef / abs(h0Ref)
pa = amp * h0 * np.exp(1j * omega0 * step)

# ##!
# h0 = hankel1(0, k * xSafe)
# pa = amp * h0 * np.exp(1j * omega * step)



In [ ]:
fig1, ax1 = plt.subplots(1, 1, figsize=(12, 3), facecolor='w', edgecolor='w')
ax1.plot(xSafe[10:], np.real(pa)[10:], c='k', linestyle='-', lw=2, alpha=1.0, label=rf'Analytical ($x_r={xRef}$)')

for case in range(0, len(folderVec)):
    ax1.plot(coordVecVec[case][:], dataVecVec[case][:], c=colorVec[case], marker='o', ms=6, fillstyle='none', lw=0, label=labelVec[case])

ax1.set_title(rf'$\tau={tau:.4f}$ $(\nu={vis:.4f})$ $T={int(1/f0)}$ ($step={int(step*f0)}T$)', loc='right')
ax1.legend(loc='best', frameon=False, fontsize=13)
ax1.set_xlim([0, x[-1]])
ax1.set_ylim([-0.001, 0.001])
# ax1.set_ylim([-0.0006, 0.0006])
ax1.set_xlabel(r'$x$')
ax1.set_ylabel(r'$\rho^\prime$')
